# BronzeWork Incremental
**Incremental Bronze ingestion with rerun-safe watermark logic**

# Setup 1-- Imports and setup
**This cell imports the pyspark and delta helpers used in notebook,switches to correct catlog,and makes sure the ** Bronze schema exists before start loading data****

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import uuid


In [0]:
spark.sql("use catalog venk_novacart")

In [0]:
spark.sql("create schema if not exists bronze_schema")

## Step -2 Bronze control table
This table stores the **watermark** for each source table

its help the pipeline remember:
- the latest timestamp already processed
- the latest timestamp key processed at that timestamp
- how many rows were written in the last run

This is what makes the Bronze load incremental and rerun safe.

In [0]:
spark.sql("""
          create table if not exists bronze_schema.ingestion_control(
            layer string,
            table_name string,
            ts_col string,
            pk_col string,
            last_successful_ts timestamp,
            last_successful_pk bigint,
            last_run_id string,
            rows_written bigint,
            run_status string,
            updated_at timestamp
            )
            using delta
          """
          )

# step 3 -- source table configuration

This cell defines which tables will be loaded into bronze which columns should be used as
- primary key
- timestamp/watermark column

its also create unique **bronze_run_id** for current pipeline run. 

In [0]:
tables_config ={
    "orders" : {"pk_col" : "order_id", "ts_col": "updated_at"},
    "products":{"pk_col" : "product_id", "ts_col": "updated_at"},
    "payments":{"pk_col" : "payment_id", "ts_col": "processed_at"}}

bronze_run_id = str(uuid.uuid4())
print("bronze_Run_ID:",bronze_run_id)

# Step 4 -- Helper functions

This cell contains reusable functions

- get_last_sucessful_watermark() reads the last processed watermark from the control table
- upsert_bronze_control() updates the control table after a sucessfull Bronze load

These functions keep the main load logic cleaner and easier to understand

In [0]:


def get_last_sucessful_watermark(table_name: str):
    ctrl = (
        spark.table("venk_novacart.bronze_schema.ingestion_control")
            .filter(
            (F.col("layer") == "bronze") &
            (F.col("table_name") == table_name) &
            (F.col("run_status") == "success")
        )
       
        .orderBy(F.col("updated_at").desc())
        .limit(1) 
    )
    rows = ctrl.collect()
    if not rows:
        return None, None

    # Now this is safe from the PySparkValueError
    return rows[0]["last_successful_ts"], rows[0]["last_successful_pk"]

In [0]:

def upsert_bronze_control(table_name, ts_col, pk_col, last_ts, last_pk, rows_written, run_id):
    # Create the DataFrame to be merged
    control_df = spark.createDataFrame(
        [(
            "bronze",
            table_name,
            ts_col,
            pk_col, 
            last_ts,
            int(last_pk) if last_pk is not None else None,
            run_id,
            int(rows_written),
            "success",
            datetime.utcnow()
        )], 
        schema="""
            layer string,
            table_name string,
            ts_col string, 
            pk_col string,
            last_successful_ts timestamp, 
            last_successful_pk bigint,
            last_run_id string, 
            rows_written bigint,
            run_status string, 
            updated_at timestamp
        """
    )

    dt = DeltaTable.forName(spark, "venk_novacart.bronze_schema.ingestion_control")
    
    (dt.alias("t")
        .merge(control_df.alias("s"), "t.table_name = s.table_name and t.layer = s.layer")
        .whenMatchedUpdate(set={
            "ts_col": "s.ts_col",
            "pk_col": "s.pk_col", 
            "last_successful_ts": "s.last_successful_ts", 
            "last_successful_pk": "s.last_successful_pk", 
            "last_run_id": "s.last_run_id",
            "rows_written": "s.rows_written",
            "run_status": "s.run_status",
            "updated_at": "s.updated_at"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )
    

        
   

# Step 5 -- Bronze incremental load loop
This is the main Bronze logic.

- -1.reads the last watermark
- -2.reads the source Sql table
- -3.filter only **new/changed rows**
- -4.adds Bronze audit columns
- -5.appends the rows into the Bronze Delta table
- -6.updates the control table

This is core incremental loading logic

In [0]:
for table_name, cfg in tables_config.items():
    pk_col = cfg["pk_col"]
    ts_col = cfg["ts_col"]

    source_table = f"`v_novacart_sql_connection_catalog`.dbo.{table_name}"
    target_table = f"venk_novacart.bronze_schema.{table_name}_raw"

    last_successful_ts, last_successful_pk = get_last_sucessful_watermark(table_name)

    print(f"\n=== processing {table_name} ===")
    print(f"last_successful_ts: {last_successful_ts}")
    print(f"last_successful_pk: {last_successful_pk}")

    source_df = spark.read.table(source_table).withColumn(ts_col, F.col(ts_col).cast("timestamp"))

    #  Incremental logic
    if last_successful_ts is None:
        rows_to_load = source_df
    else:
        # Note: ensuring last_successful_pk can be cast to int safely
        #safe_pk = int(last_successful_pk) if last_successful_pk is not None else 0
        rows_to_load = source_df.filter(
            (F.col(ts_col) > F.lit(last_successful_ts)) |
            (
                (F.col(ts_col) == F.lit(last_successful_ts)) &
                (F.col(pk_col).cast("long") > F.lit(int(last_successful_pk)))
             )
        )

    # Metadata columns
    rows_to_load = (
        rows_to_load
        .withColumn("bronze_ingested_at", F.current_timestamp())
        .withColumn("bronze_run_id", F.lit(bronze_run_id))
        .withColumn("bronze_source_table", F.lit(source_table))
    )

    rows_count = rows_to_load.count()
    print(f"{table_name} rows to load: {rows_count}")

    # Handle no data case
    if rows_count == 0:
        print(f"No new data to load for {table_name}")
        upsert_bronze_control(
            table_name,
            ts_col,
            pk_col,
            last_successful_ts,
            last_successful_pk,
            rows_count,
            bronze_run_id
        )
        continue

    #  WRITE
    rows_to_load.write.format("delta").mode("append").saveAsTable(target_table)
    
    max_ts = rows_to_load.agg(F.max(ts_col).alias("max_ts")).collect()[0]["max_ts"]
    

    #  Get max PK for that timestamp
    max_pk = (
        rows_to_load
        .filter(F.col(ts_col) == F.lit(max_ts))
        .agg(F.max(F.col(pk_col).cast("long")).alias("max_pk"))
        .collect()[0]["max_pk"]
    )

    #  Update control table
    upsert_bronze_control(
        table_name,
        ts_col,
        pk_col,
        max_ts,
        max_pk,
        rows_count,
        bronze_run_id
    )

    print(f"Wrote {rows_count} rows to {target_table}")

# Step 6 -- Quick validation

This final cell prints the Bronze row counts and display the control table so you can verify that the incremental logic is working correctly 

In [0]:
print("orders Bronze Count:",spark.table("venk_novacart.bronze_schema.orders_raw").collect()[0][0])
print("products Bronze Count:",spark.table("venk_novacart.bronze_schema.products_raw").collect()[0][0])
print("payments Bronze Count:",spark.table("venk_novacart.bronze_schema.payments_raw").collect()[0][0])

display(spark.sql("select * from venk_novacart.bronze_schema.ingestion_control").orderBy("table_name"))